In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Connect to Rent database & load data

Loading Rent_Stage_1 dataset to dataframe and create a copy.

In [2]:
conn = sqlite3.connect('Rent.db')

In [3]:
query = "SELECT * FROM Rent_Stage_1"
df_raw = pd.read_sql(query,conn)

In [4]:
df = df_raw.copy()

# 2. Dataset overview & Summary Statistics

Perform an initial inspection of the dataset's structure, data types, and generate descriptive statiscs for both numeric and categorical

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70847 entries, 0 to 70846
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    70847 non-null  object 
 1   date                  70847 non-null  object 
 2   city                  70847 non-null  object 
 3   type                  54085 non-null  object 
 4   squareMeters          70847 non-null  float64
 5   rooms                 70847 non-null  float64
 6   floor                 62234 non-null  float64
 7   floorCount            69468 non-null  float64
 8   buildYear             51165 non-null  float64
 9   latitude              70847 non-null  float64
 10  longitude             70847 non-null  float64
 11  centreDistance        70847 non-null  float64
 12  poiCount              70847 non-null  float64
 13  schoolDistance        70832 non-null  float64
 14  clinicDistance        70797 non-null  float64
 15  postOfficeDistance 

**Insight:** column date has wrong type I must change to datatime

In [6]:
df['date'] = pd.to_datetime(df['date'])

In [7]:
df.shape

(70847, 28)

In [8]:
df.describe()

,date,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,centreDistance,poiCount,schoolDistance,clinicDistance,postOfficeDistance,kindergartenDistance,restaurantDistance,collegeDistance,pharmacyDistance,price,price_per_m2
count,70847,70847.000000,70847.000000,62234.000000,69468.000000,51165.000000,70847.000000,70847.000000,70847.000000,70847.000000,70832.000000,70797.000000,70821.000000,70804.000000,70693.000000,69973.000000,70767.000000,70847.000000,70847.000000
mean,2024-02-17 01:21:24.215280640,55.252692,2.384942,3.419787,5.686906,1998.109293,51.785566,19.691005,3.856228,25.207983,0.368818,0.776294,0.470210,0.327194,0.260977,1.286804,0.308887,3852.796886,69.203438
min,2023-11-01 00:00:00,25.000000,1.000000,1.000000,1.000000,1850.000000,49.982150,14.447100,0.020000,0.000000,0.002000,0.001000,0.001000,0.001000,0.001000,0.004000,0.001000,346.000000,11.300000
25%,2024-01-01 00:00:00,40.000000,2.000000,2.000000,4.000000,1984.000000,51.078600,18.651526,1.770000,9.000000,0.174000,0.302000,0.235000,0.153000,0.087000,0.520000,0.136000,2500.000000,53.000000
50%,2024-03-01 00:00:00,50.000000,2.000000,3.000000,5.000000,2010.000000,52.185642,19.958724,3.380000,17.000000,0.285000,0.550000,0.388000,0.259000,0.180000,0.971000,0.231000,3100.000000,65.960000
75%,2024-05-01 00:00:00,64.000000,3.000000,4.000000,7.000000,2020.000000,52.250101,21.005240,5.420000,30.000000,0.449000,0.970000,0.585000,0.401000,0.329000,1.809000,0.382000,4490.000000,81.580000
max,2024-06-01 00:00:00,150.000000,6.000000,30.000000,30.000000,2024.000000,54.581756,23.199087,16.620000,210.000000,4.856000,4.990000,4.939000,4.751000,4.961000,5.000000,4.986000,23000.000000,189.470000
std,NaN,22.688973,0.872078,2.647224,3.461192,30.292490,1.183777,1.675260,2.609485,27.050432,0.341474,0.724202,0.389788,0.321697,0.318860,1.016936,0.327246,2381.540981,23.349050


**Insights:** Analyzing the table, we can notice potential outliers in the **price** column, where the minimum value is 346 PLN, and in the **price_per_m2 column**, where the minimum value is 11.3 PLN. We will address these values during the EDA stage. The remaining values appear to be realistic.

In [9]:
df.describe(include='object')

,id,city,type,ownership,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom
count,70847,70847,54085,70847,70847,70847,66923,70847,70847
unique,37941,15,3,1,2,2,2,2,2
top,f669872f67ece87f8e8fce68ac27f31a,warszawa,apartmentBuilding,condominium,no,yes,yes,no,no
freq,8,28414,23910,70847,43075,43506,44063,60614,59833


# 3. Automated Data Quality Audit

Design a custom, resuable Python class 'DataQualityChecker' to systematically identify missing values and track duplicate records across the dataset.

In [10]:
class DataQualityChecker:
    def __init__(self,dataframe):
        self.df = dataframe
    def check_missing(self):
        missing_value_pct = ((self.df.isnull().sum()) / len(self.df)) * 100
        return missing_value_pct[missing_value_pct > 0].sort_values(ascending=False).round(5)
    def check_duplicates(self):
        duplicated = self.df.duplicated().sum()
        print(f"Total duplicate rows in dataset: {duplicated}")

In [11]:
check = DataQualityChecker(df)

In [12]:
check.check_missing()

buildYear               27.78099
type                    23.65944
floor                   12.15718
hasElevator              5.53870
floorCount               1.94645
collegeDistance          1.23364
restaurantDistance       0.21737
pharmacyDistance         0.11292
clinicDistance           0.07057
kindergartenDistance     0.06069
postOfficeDistance       0.03670
schoolDistance           0.02117
dtype: float64

**Insights:** In this dataset, we have 12 columns with missing values. In the next step, we will impute these values or drop the affected records.

In [13]:
check.check_duplicates()

Total duplicate rows in dataset: 0


**Insights:** We found no duplicated records in this dataset.

# 4. Advanced Data Imputation & Cleaning

Design a custom, resuable Python class 'DataInputer' which contains custom methods for missing data imputation.

In [14]:
((df['type'].isna()) & (df['buildYear'].isna())).sum()

np.int64(7736)

**Insight:** I got 7736 records with missing value in two both columns.

 Below is a breakdown of these methods and their underlying logic:
* **distance_values** - This method handles missing values in distance-related columns by applying city-level median.
*  **floorCount** - Fills missing values with the city-level-median, followed by the global median to ensure no nulls are left behind.
*  **elevator** - Uses domain knowledge (Polish building law) to fill values: assign 'no' to buildings with <=4 florrs and 'yes' to those with > 4 floors. Any leftover nulls are defaulted to 'no'
*  **type** - This method applies a robust 3-stage imputation strategy. **Stage 1** uses domain knowledge regarding historical architectural periods in Poland to infer the building type based on the construction year. **Stage 2** targests 7736 records missing both **type** and **buildYear** by applying the group-level-mode based on **city** and *floorCount**. **Stage 3** acts as a final fallback, filling any remaining nulls with global mode.
*  **year** - This method imputes missing construction years by calculating the group-level-median based on the **city** and **type** columns , ensuring architectural and regional consistency.
*  **floor** - Applies a two-step grouped imputation strategy. It first calculates the median floor based on **city**, **type**, and **floorCount**. Any residual missing values are filled using the overall **floorCount** median. Finally, all values are rounded to preserve discrete integer types.

In [27]:
class DataImputer:
    def __init__(self, dataframe):
        self.df = dataframe.copy()
    def distance_values(self,column):
        median_city = self.df.groupby('city')[column].transform('median')
        self.df[column] = self.df[column].fillna(median_city)
    def floorCount(self,column):
        median_floorCount = self.df.groupby('city')[column].transform('median')
        self.df[column] = self.df[column].fillna(median_floorCount)
        if self.df[column].isna().any():
            self.df[column] = self.df[column].fillna(self.df[column].median())
    def elevator(self,column):
        self.df.loc[(self.df['floorCount'] <= 4) & (self.df[column].isna()), column] = 'no'
        self.df.loc[(self.df['floorCount'] > 4) & (self.df[column].isna()), column] = 'yes'
        if self.df[column].isna().any():
            self.df[column] = self.df[column].fillna('no')
    def type(self,column):
        def _impute_row(row):
            if pd.notna(row[column]):
                return row[column]
            if pd.notna(row['buildYear']):
                year = row['buildYear']
                if year < 1950:
                    return 'tenement'
                elif 1950 <= year <=2010 :
                    return 'blockOfFlats'
                elif year > 2010:
                    return 'tenement'
            return row[column]
        self.df[column] = self.df.apply(_impute_row, axis=1)
        self.df[column] = self.df[column].fillna(self.df.groupby(['city', 'floorCount'])[column].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan))
        self.df[column] = self.df[column].fillna(self.df[column].mode()[0])
    def year(self,column):
        self.df[column] = self.df[column].fillna(self.df.groupby(['city','type'])[column].transform('median'))
    def floor(self,column):
        self.df[column] = self.df[column].fillna(self.df.groupby(['city','type','floorCount'])[column].transform('median'))
        self.df[column] = self.df[column].fillna(self.df.groupby('floorCount')[column].transform('median'))
        self.df[column] = self.df[column].round()
        

In [31]:
columns = ['kindergartenDistance','clinicDistance','pharmacyDistance','collegeDistance','restaurantDistance','postOfficeDistance','schoolDistance']
imputer = DataInputer(df)
for c in columns:
    inputer.distance_values(c)
imputer.floorCount('floorCount')
imputer.elevator('hasElevator')
imputer.type('type')
imputer.year('buildYear')
imputer.floor('floor')
check = DataQualityChecker(df)
check.check_missing()

Series([], dtype: float64)

**Insights:** All data imputed.

In [32]:
df = imputer.df